In [ ]:
import pandas as pd
import os
import cv2

In [ ]:
def images_to_video(image_folder: str, 
                   start_frame: int, 
                   end_frame: int, 
                   output_file: str, 
                   fps: int = 10) -> None:
    """
    Convert a sequence of images to video with frame range expansion.
    """
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    # Adjust frame range to include buffer
    start_frame = max(0, start_frame - 5)
    end_frame = end_frame + 5

    # Get image files for the specified range
    images = []
    current_frame = start_frame
    while current_frame <= end_frame:
        image_path = os.path.join(image_folder, f"output_{current_frame}.jpg")
        if os.path.exists(image_path):
            images.append(image_path)
        current_frame += 1

    if not images:
        print("No images found in the specified range.")
        return

    # Read first image to get dimensions
    frame = cv2.imread(images[0])
    height, width, layers = frame.shape

    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_file, fourcc, fps, (width, height))

    # Write frames to video
    for image_path in images:
        video.write(cv2.imread(image_path))

    # Release resources
    cv2.destroyAllWindows()
    video.release()

    print(f"Video saved as {output_file}")

In [ ]:
def analyze_results(detected_frames_csv, ground_truth_csv, image_folder, output_folder):    
    """
    Compare detection results with ground truth by matching ground truth frames to detection ranges.
    
    Parameters:
    -----------
    detected_frames_csv : str
        Path to CSV containing detected passing events with frame ranges
    ground_truth_csv : str
        Path to CSV containing ground truth frame numbers
    image_folder : str
        Path to folder containing source images
    output_folder : str
        Path to save analysis results and video clips
    """
    
    # Create output folders
    fp_folder = os.path.join(output_folder, "false_positives")
    fn_folder = os.path.join(output_folder, "false_negatives")
    os.makedirs(fp_folder, exist_ok=True)
    os.makedirs(fn_folder, exist_ok=True)

    # Load and preprocess data
    detected_frames = pd.read_csv(detected_frames_csv, header=0, 
                                names=['overtaking_id', 'track_id', 'first_frame', 
                                      'last_frame', 'vehicle_class'])
    ground_truth = pd.read_csv(ground_truth_csv, header=0, 
                             names=['overtaking_frame', 'lane'])
    
    # Convert to integer and sort by end_frame for detections
    detected_frames = detected_frames.sort_values('last_frame')
    ground_truth['overtaking_frame'] = ground_truth['overtaking_frame'].astype(int)
    ground_truth = ground_truth.sort_values('overtaking_frame')
    
    # Create copies for iteration to allow removal of matched items
    remaining_detections = detected_frames.copy()
    remaining_gt = ground_truth.copy()
    
    # Initialize tracking variables
    matched_pairs = []
    false_positives = []
    false_negatives = []

    # First pass: Match ground truth frames to detections
    while len(remaining_gt) > 0:
        gt_row = remaining_gt.iloc[0]
        gt_frame = gt_row['overtaking_frame']
        match_found = False
        
        # Find the best matching detection for this ground truth frame
        for idx, detection in remaining_detections.iterrows():
            if (detection['first_frame'] <= gt_frame <= detection['last_frame']):
                matched_pairs.append({
                    'gt_frame': gt_frame,
                    'detection_id': detection['overtaking_id'],
                    'track_id': detection['track_id'],
                    'start_frame': detection['first_frame'],
                    'end_frame': detection['last_frame']
                })
                
                remaining_detections = remaining_detections.drop(idx)
                remaining_gt = remaining_gt.drop(gt_row.name)
                match_found = True
                break
        
        if not match_found:
            false_negatives.append(gt_frame)
            remaining_gt = remaining_gt.drop(gt_row.name)

    # Mark remaining detections as potential false positives
    potential_fps = []
    for _, detection in remaining_detections.iterrows():
        potential_fps.append({
            'overtaking_id': detection['overtaking_id'],
            'track_id': detection['track_id'],
            'start_frame': detection['first_frame'],
            'end_frame': detection['last_frame']
        })

    # Second pass: Cross-check FPs and FNs
    revised_fps = []
    revised_fns = []
    
    for fp in potential_fps:
        fp_matched = False
        for fn_frame in false_negatives[:]:  # Use slice copy to modify original list
            # Check if false negative frame is within ±20 frames of the detection
            if (fp['start_frame'] - 20 <= fn_frame <= fp['end_frame'] + 20):
                # This is actually a match with slight timing difference
                matched_pairs.append({
                    'gt_frame': fn_frame,
                    'detection_id': fp['overtaking_id'],
                    'track_id': fp['track_id'],
                    'start_frame': fp['start_frame'],
                    'end_frame': fp['end_frame']
                })
                false_negatives.remove(fn_frame)
                fp_matched = True
                break
        
        if not fp_matched:
            revised_fps.append(fp)

    # Generate videos for final FPs and FNs
    for fp in revised_fps:
        output_file = os.path.join(fp_folder, 
            f'fp_id_{fp["overtaking_id"]}_frames_{fp["start_frame"]}_{fp["end_frame"]}.mp4')
        images_to_video(image_folder, fp['start_frame'], fp['end_frame'], output_file)

    for fn_frame in false_negatives:
        output_file = os.path.join(fn_folder, f'fn_frame_{fn_frame}.mp4')
        start_frame = max(1, fn_frame - 20)
        last_frame = fn_frame + 20
        images_to_video(image_folder, start_frame, last_frame, output_file)

    # Calculate final metrics
    total_gt = len(ground_truth)
    total_detections = len(detected_frames)
    true_positives = len(matched_pairs)
    false_positives_count = len(revised_fps)
    false_negatives_count = len(false_negatives)
    
    precision = true_positives / total_detections if total_detections > 0 else 0
    recall = true_positives / total_gt if total_gt > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Prepare results dictionary
    # Update the results dictionary to use revised_fps instead of false_positives
    results = {
        'metrics': {
            'precision': precision,
            'recall': recall,
            'f1_score': f1_score,
            'total_ground_truth': total_gt,
            'total_detections': total_detections,
            'true_positives': true_positives,
            'false_positives': false_positives_count,
            'false_negatives': false_negatives_count
        },
        'matched_pairs': matched_pairs,
        'false_positives': revised_fps,  # Changed from false_positives to revised_fps
        'false_negatives': false_negatives
    }

    # Export results to CSV files
    # Export matched pairs
    matched_pairs_df = pd.DataFrame(matched_pairs)
    matched_pairs_df = matched_pairs_df.sort_values('gt_frame')
    matched_pairs_csv = os.path.join(output_folder, 'matched_pairs.csv')
    # matched_pairs_df.to_csv(matched_pairs_csv, index=False)

    # Export false positives
    false_positives_df = pd.DataFrame(revised_fps)  # Changed from false_positives to revised_fps
    false_positives_csv = os.path.join(output_folder, 'false_positives.csv')
    # false_positives_df.to_csv(false_positives_csv, index=False)

    # Export false negatives
    false_negatives_df = pd.DataFrame(false_negatives, columns=['frame'])
    false_negatives_csv = os.path.join(output_folder, 'false_negatives.csv')
    # false_negatives_df.to_csv(false_negatives_csv, index=False)

    # Export summary metrics
    metrics_df = pd.DataFrame([results['metrics']])
    metrics_csv = os.path.join(output_folder, 'metrics_summary.csv')
    # metrics_df.to_csv(metrics_csv, index=False)

        # Print analysis results with modified output
    print("\n=== Detection Performance Analysis ===")
    print(f"Ground Truth Events: {total_gt}")
    print(f"Total Detections: {total_detections}")
    print(f"True Positives: {true_positives}")
    print(f"False Positives: {false_positives_count}")
    print(f"False Negatives: {false_negatives_count}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")
    
    # Print detailed false positives information
    if false_positives_count > 0:
        print("\nFalse Positive Details:")
        for fp in revised_fps:
            print(f"ID: {fp['overtaking_id']}, Track: {fp['track_id']}, "
                  f"Frames: {fp['start_frame']}-{fp['end_frame']}")

    print("\nResults exported to CSV files in:", output_folder)
    print(f"- False positives: {false_positives_csv}")
    print(f"- False negatives: {false_negatives_csv}")
    # print(f"- Metrics summary: {metrics_csv}")
    
    return results

In [ ]:
# Trip 1 Analysis IR
analyze_results(
    detected_frames_csv='./results/trip_1_ir/vehicle_passing.csv',
    ground_truth_csv='./data/ground_truth_annotations/trip1/ground_truth_2023_07_26.csv',
    image_folder='./results/trip_1_ir/inference_images/',
    output_folder='./results/trip_1_ir/analysis/'
)

In [ ]:
# Trip 2 Analysis IR
analyze_results(
    detected_frames_csv='./results/trip_2_ir/vehicle_passing.csv',
    ground_truth_csv='./data/ground_truth_annotations/trip2/ground_truth_2023_08_04.csv',
    image_folder='./results/trip_2_ir/inference_images/',
    output_folder='./results/trip_2_ir/analysis/'
)

In [ ]:
# Trip 1 Analysis GoPro Rear-Left
analyze_results(
    detected_frames_csv='./results/trip_1_gopro_rl/vehicle_passing.csv',
    ground_truth_csv='./data/ground_truth_annotations/trip1/ground_truth_2023_07_26.csv',
    image_folder='./results/trip_1_gopro_rl/inference_images/',
    output_folder='./results/trip_1_gopro_rl/analysis/'
)

In [ ]:
analyze_results(
    detected_frames_csv='./results/trip_2_gopro_rl/vehicle_passing.csv',
    ground_truth_csv='./data/ground_truth_annotations/trip2/ground_truth_2023_08_04.csv',
    image_folder='./results/trip_2_gopro_rl/inference_images/',
    output_folder='./results/trip_2_gopro_rl/analysis/'
)

In [ ]:
images_to_video('./results/trip_1_ir/inference_images/', 0, 16439, './results/trip_1_ir/demo_fulltrip1_ir.mp4')

In [ ]:
images_to_video('./results/trip_1_gopro_rl/inference_images/', 0, 16426, './results/trip_1_gopro_rl/demo_fulltrip1_gopro_rl.mp4')

In [ ]:
images_to_video('./results/trip_2_ir/inference_images/', 0, 25000, './results/trip_2_ir/demo_fulltrip2_ir.mp4')

In [ ]:
images_to_video('./results/trip_2_gopro_rl/inference_images/', 0, 25000, './results/trip_2_gopro_rl/demo_fulltrip2_gopro_rl.mp4')